# 03 训练 PoseCorrector 模型

使用 **5 件套 Loss**（L_pose, L_Δ, L_smooth, L_bone, L_id）训练 Corrector，输出修正位移 Δ，y_hat = x + Δ。

**数据**：由 `01_data_prep.ipynb` 生成的 `.npz`，含 `landmarks`, `phase`, `corrected`, `visibility`, `confidence`。


In [ ]:
# 环境与路径
from pathlib import Path
import sys

PROJECT_ROOT = Path("/content/ai_form_1")
DATA_ROOT = Path("/content/drive/MyDrive/Colab_Notebooks_AIForm/Model20260305/ai_form_coach_data")
NPZ_PATH = DATA_ROOT / "train_data.npz"
if not NPZ_PATH.exists():
  NPZ_PATH = PROJECT_ROOT / "tools" / "train_denoiser" / "data" / "train_data.npz"
CKPT_DIR = DATA_ROOT / "checkpoints"
if not Path("/content/drive").exists():
  CKPT_DIR = PROJECT_ROOT / "tools" / "train_denoiser" / "checkpoints"

sys.path.insert(0, str(PROJECT_ROOT / "tools" / "train_denoiser" / "src"))

from model import PoseCorrectorNet
from train import make_dataloaders, train_one_epoch, save_checkpoint

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


In [ ]:
# Colab: 挂载 Drive
try:
  from google.colab import drive
  drive.mount("/content/drive")
except ImportError:
  pass

In [ ]:
# 构建 DataLoader
loaders = make_dataloaders(NPZ_PATH, batch_size=128, val_ratio=0.1)
print("Train batches:", len(loaders["train"]), "| Val batches:", len(loaders["val"]))

In [ ]:
# 模型与优化器
model = PoseCorrectorNet(hidden_dim=256, num_layers=4).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
# 训练循环（5 件套 loss）
NUM_EPOCHS = 50
history = []

for epoch in range(NUM_EPOCHS):
  stats = train_one_epoch(
    model, loaders, optimizer, device,
    lambda_pose=1.0,
    lambda_delta=0.1,
    lambda_smooth=0.1,
    lambda_bone=0.2,
    lambda_id=0.2,
  )
  history.append(stats)
  if (epoch + 1) % 10 == 0 or epoch == 0:
    print(f"Epoch {epoch+1:3d} | train_loss={stats['train_loss']:.4f} | val_loss={stats['val_loss']:.4f}")

In [ ]:
# 绘制 loss 曲线
import matplotlib.pyplot as plt

epochs = range(1, len(history) + 1)
plt.figure(figsize=(8, 4))
plt.plot(epochs, [h["train_loss"] for h in history], label="train")
plt.plot(epochs, [h["val_loss"] for h in history], label="val")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# 保存最佳 checkpoint（供 04_export_onnx 使用）
CKPT_DIR.mkdir(parents=True, exist_ok=True)
best_epoch = min(range(len(history)), key=lambda i: history[i]["val_loss"])
save_checkpoint(model, optimizer, best_epoch, history[best_epoch], CKPT_DIR / "pose_corrector_best.pt")
print(f"已保存最佳模型 (epoch {best_epoch+1}) 到 {CKPT_DIR}")

In [ ]:
# 动态可视化：训练样本 vs ML 纠错（3D 动画）
import numpy as np
from matplotlib import pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML, display


# 加载最佳 checkpoint
ckpt = torch.load(CKPT_DIR / "pose_corrector_best.pt", map_location=device)
model.load_state_dict(ckpt["model_state"])
model.eval()

data = np.load(NPZ_PATH)
lm_all = data["landmarks"]
ph_all = data["phase"]
if ph_all.ndim == 1:
  ph_all = ph_all.reshape(-1, 1)

# 取一段连续帧（模拟一个 rep 的动作序列）
# 数据按 augment_per_frame 排列，每 3 帧来自同一 rep 的同一时刻，取步长 3 得连续动作
step = 3
start = np.random.randint(0, max(1, len(lm_all) - 80 * step))
seq_len = min(80, (len(lm_all) - start) // step)
idx = np.arange(start, start + seq_len * step, step)

lm_seq = lm_all[idx]
ph_seq = ph_all[idx]

with torch.no_grad():
  lmk_t = torch.from_numpy(lm_seq).float().to(device)
  ph_t = torch.from_numpy(ph_seq).float().to(device)
  delta, _ = model(lmk_t, ph_t)
  pred_seq = (lmk_t + delta).cpu().numpy()

EDGES = [(11, 13), (13, 15), (12, 14), (14, 16), (23, 11), (24, 12), (23, 24), (23, 25), (24, 26), (25, 27), (26, 28)]

fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection="3d")

def init():
  ax.clear()
  return []

def update(t):
  ax.clear()
  user = lm_seq[t]
  pred = pred_seq[t]
  for i, j in EDGES:
    ax.plot([user[i, 0], user[j, 0]], [user[i, 1], user[j, 1]], [user[i, 2], user[j, 2]],
            color="black", linewidth=2, alpha=0.7)
  ax.scatter(user[:, 0], user[:, 1], user[:, 2], color="black", s=15, alpha=0.7, label="User")
  for i, j in EDGES:
    ax.plot([pred[i, 0], pred[j, 0]], [pred[i, 1], pred[j, 1]], [pred[i, 2], pred[j, 2]],
            color="#ffb432", linewidth=2, alpha=0.8)
  ax.scatter(pred[:, 0], pred[:, 1], pred[:, 2], color="#ffb432", s=15, alpha=0.8, label="ML correction")
  ax.set_title(f"Frame {t}/{seq_len} | phase={ph_seq[t].item():.2f}")
  ax.legend()
  ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_zlabel("Z")
  return []

anim = FuncAnimation(fig, update, frames=seq_len, init_func=init, interval=80, blit=False)

# 在 Notebook 中播放（可点击播放/暂停）
display(HTML(anim.to_jshtml()))

# 导出 GIF 到 Drive
PLOT_DIR = CKPT_DIR / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)
gif_path = PLOT_DIR / "compare_animation.gif"
anim.save(str(gif_path), writer=PillowWriter(fps=12))
print(f"已保存动态 GIF 到 {gif_path}")